# HUC8 Water-Stress GitHub Demo — Notebook v4

This notebook provides a computation demonstration of the project water-stress workflow. It loads the data resources, simulates monthly freshwater stress for each HUC8 under three SSP–RCP pathways and four AI cases, and prints selected national and HUC8-level results.

In [1]:
"""
GitHub demo notebook for monthly HUC8 freshwater-stress simulation.

The workflow follows the original project notebook but contains no plotting,
GIS rendering, command-line parser, or file-output operations. Running the
final notebook cell simulates all SSP-RCP and AI cases and prints examples.
"""

from __future__ import annotations

import glob
from collections import OrderedDict
from pathlib import Path
from typing import Dict, Iterable, List, Mapping, Optional, Sequence, Tuple

import numpy as np
import pandas as pd


# =============================================================================
# NOTEBOOK CONFIGURATION
# =============================================================================
# Replace with data loading path
BASE_PATH = "Data Load Path"

# Detailed HUC8 output to print after all scenario/case simulations finish.
SELECTED_SCENARIO = "SSP370"       # SSP245, SSP370, or SSP585
SELECTED_AI_CASE = "Mid-Case"      # Baseline, Low-Case, Mid-Case, or High-Case
SELECTED_YEAR = 2035                # 2020, 2025, 2030, or 2035
TOP_N = 10                          # Number of ranked HUC8 rows to print
REQUESTED_HUC8 = None               # Example: "01090005"; use None to skip


# =============================================================================
# Model settings
# =============================================================================
YEARS: List[int] = [2020, 2025, 2030, 2035]
MONTHS: List[int] = list(range(1, 13))
TARGET_COLS: List[str] = [f"{year}_{month:02d}" for year in YEARS for month in MONTHS]
COLS_2020: List[str] = [f"2020_{month:02d}" for month in MONTHS]

HIGH_STRESS_THRESHOLD = 0.4
OTHER_USE_FRACTION_OF_2020_BASELINE = 0.10
TREAT_NONPOSITIVE_AVAILABLE_AS_INF = False


# =============================================================================
# Scenario definitions copied from the project notebook
# =============================================================================
SCENARIOS = OrderedDict(
    {
        "SSP245": {
            "title": "SSP2-RCP4.5",
            "runoff_folder": "ssp245",
            "ir_file": "IR_HUC8_Tot_WD_monthly_2020_2035_SSP2_m3_per_month.csv",
            "ps_file": "PS_HUC8_H08_projected_SSP2_2020_2035_monthly_m3.csv",
            "tp_file": "TPWW_HUC8_SSP2_HE2_monthly_m3_2020_2025_2030_2035_with_Baseline.xlsx",
            "ai_proj_files": {
                "Low-Case": "AI_Power_Water_Usage_RCP4_5_Proj1.xlsx",
                "Mid-Case": "AI_Power_Water_Usage_RCP4_5_Proj2.xlsx",
                "High-Case": "AI_Power_Water_Usage_RCP4_5_Proj3.xlsx",
            },
        },
        "SSP370": {
            "title": "SSP3-RCP7.0",
            "runoff_folder": "ssp370",
            "ir_file": "IR_HUC8_Tot_WD_monthly_2020_2035_SSP3_m3_per_month.csv",
            "ps_file": "PS_HUC8_H08_projected_SSP3_2020_2035_monthly_m3.csv",
            "tp_file": "TPWW_HUC8_SSP3_HE2_monthly_m3_2020_2025_2030_2035_with_Baseline.xlsx",
            "ai_proj_files": {
                "Low-Case": "AI_Power_Water_Usage_RCP7_0_Proj1.xlsx",
                "Mid-Case": "AI_Power_Water_Usage_RCP7_0_Proj2.xlsx",
                "High-Case": "AI_Power_Water_Usage_RCP7_0_Proj3.xlsx",
            },
        },
        "SSP585": {
            "title": "SSP5-RCP8.5",
            "runoff_folder": "ssp585",
            "ir_file": "IR_HUC8_Tot_WD_monthly_2020_2035_SSP5_m3_per_month.csv",
            "ps_file": "PS_HUC8_H08_projected_SSP5_2020_2035_monthly_m3.csv",
            "tp_file": "TPWW_HUC8_SSP5_HE2_monthly_m3_2020_2025_2030_2035_with_Baseline.xlsx",
            "ai_proj_files": {
                "Low-Case": "AI_Power_Water_Usage_RCP8_5_Proj1.xlsx",
                "Mid-Case": "AI_Power_Water_Usage_RCP8_5_Proj2.xlsx",
                "High-Case": "AI_Power_Water_Usage_RCP8_5_Proj3.xlsx",
            },
        },
    }
)

AI_CASES = OrderedDict(
    {
        "Baseline": {"tp_sheet": "Baseline", "label": "No AI"},
        "Low-Case": {"tp_sheet": "Proj1", "label": "Low-Case AI"},
        "Mid-Case": {"tp_sheet": "Proj2", "label": "Mid-Case AI"},
        "High-Case": {"tp_sheet": "Proj3", "label": "High-Case AI"},
    }
)


## 1. Data-format helpers
These functions standardize HUC8 identifiers and monthly columns and load the CSV/XLSX inputs.

In [2]:
# =============================================================================
# Data-format helpers
# =============================================================================
def standardize_huc8(value: object) -> Optional[str]:
    """Convert a HUC8 identifier to an eight-digit string."""
    if pd.isna(value):
        return None

    text = str(value).strip()
    if text.endswith(".0"):
        text = text[:-2]
    text = text.replace("'", "").replace('"', "").strip()
    digits = "".join(character for character in text if character.isdigit())
    return digits.zfill(8) if digits else None


def detect_huc8_col(columns: Iterable[object]) -> object:
    """Locate a HUC8 column while tolerating minor naming differences."""
    columns = list(columns)
    for column in columns:
        normalized = str(column).strip().lower().replace("_", "")
        if normalized == "huc8":
            return column
    for column in columns:
        if "huc8" in str(column).strip().lower():
            return column
    raise KeyError("Could not identify a HUC8 column.")


def standardize_date_col(column: object) -> str:
    """Standardize date columns to YYYY_MM, such as 2035_01."""
    text = str(column).strip()
    if text.lower().replace("_", "") == "huc8":
        return "HUC8"

    pieces = text.replace("-", "_").replace("/", "_").split("_")
    if len(pieces) == 2 and all(piece.isdigit() for piece in pieces):
        year, month = int(pieces[0]), int(pieces[1])
        if year in YEARS and 1 <= month <= 12:
            return f"{year}_{month:02d}"
    return text


def finalize_monthly_df(
    frame: pd.DataFrame,
    *,
    fill_missing_cols: float = 0.0,
    aggregate: str = "sum",
) -> pd.DataFrame:
    """
    Return a HUC8-indexed monthly table with one consistent set of columns.

    Duplicate HUC8 rows are summed for withdrawals and averaged for runoff.
    """
    frame = frame.copy()
    huc8_col = detect_huc8_col(frame.columns)
    frame = frame.rename(columns={huc8_col: "HUC8"})
    frame.columns = [standardize_date_col(column) for column in frame.columns]
    frame["HUC8"] = frame["HUC8"].map(standardize_huc8)
    frame = frame.loc[frame["HUC8"].notna()].copy()

    available_cols = [column for column in TARGET_COLS if column in frame.columns]
    frame = frame[["HUC8", *available_cols]].copy()

    for column in available_cols:
        frame[column] = pd.to_numeric(frame[column], errors="coerce")

    for column in TARGET_COLS:
        if column not in frame.columns:
            frame[column] = fill_missing_cols

    frame = frame[["HUC8", *TARGET_COLS]]

    if aggregate == "sum":
        frame = frame.groupby("HUC8", as_index=False)[TARGET_COLS].sum(min_count=1)
    elif aggregate == "mean":
        frame = frame.groupby("HUC8", as_index=False)[TARGET_COLS].mean()
    else:
        raise ValueError("aggregate must be either 'sum' or 'mean'.")

    return frame.set_index("HUC8").sort_index()


def read_csv_monthly(path: Path) -> pd.DataFrame:
    """Load a monthly withdrawal CSV."""
    return finalize_monthly_df(pd.read_csv(path), fill_missing_cols=0.0, aggregate="sum")


def read_excel_monthly(
    path: Path,
    sheet_name: str,
    *,
    fill_missing_cols: float = 0.0,
    aggregate: str = "sum",
) -> pd.DataFrame:
    """Load one monthly worksheet."""
    return finalize_monthly_df(
        pd.read_excel(path, sheet_name=sheet_name),
        fill_missing_cols=fill_missing_cols,
        aggregate=aggregate,
    )


def find_runoff_sheet(path: Path) -> str:
    """Find the worksheet whose name contains 'runoff'."""
    workbook = pd.ExcelFile(path)
    for sheet_name in workbook.sheet_names:
        if "runoff" in str(sheet_name).strip().lower():
            return sheet_name
    raise ValueError(f"No runoff worksheet found in {path}")


def read_runoff_file(path: Path) -> pd.DataFrame:
    """Load one runoff-model workbook and average duplicate HUC8 rows."""
    return read_excel_monthly(
        path,
        find_runoff_sheet(path),
        fill_missing_cols=np.nan,
        aggregate="mean",
    )


def average_df_list(frames: Sequence[pd.DataFrame]) -> pd.DataFrame:
    """Average aligned HUC8 tables while ignoring missing values."""
    if not frames:
        raise ValueError("At least one dataframe is required for averaging.")

    union_index = sorted(set().union(*(frame.index for frame in frames)))
    aligned = [frame.reindex(index=union_index, columns=TARGET_COLS) for frame in frames]
    combined = pd.concat(aligned, keys=range(len(aligned)), axis=0)
    averaged = combined.groupby(level=1).mean()
    averaged.index.name = "HUC8"
    return averaged.sort_index()


def align_to_index(
    frame: pd.DataFrame,
    union_index: Sequence[str],
    *,
    fill_value: Optional[float],
) -> pd.DataFrame:
    """Align a monthly table to the complete HUC8 universe."""
    aligned = frame.reindex(index=union_index, columns=TARGET_COLS)
    return aligned if fill_value is None else aligned.fillna(fill_value)


## 2. Water-stress calculations
These functions calculate monthly and annual HUC8 stress.

In [3]:
# =============================================================================
# Stress calculations
# =============================================================================
def compute_stress(withdrawal: pd.DataFrame, availability: pd.DataFrame) -> pd.DataFrame:
    """Calculate monthly HUC8 freshwater stress."""
    numerator = withdrawal.astype(float)
    denominator = availability.astype(float)
    stress = pd.DataFrame(index=numerator.index, columns=numerator.columns, dtype=float)

    positive_availability = denominator > 0
    stress[positive_availability] = numerator[positive_availability] / denominator[positive_availability]

    nonpositive_availability = ~positive_availability
    if TREAT_NONPOSITIVE_AVAILABLE_AS_INF:
        stress[nonpositive_availability & (numerator > 0)] = np.inf
        stress[nonpositive_availability & (numerator <= 0)] = 0.0
    else:
        stress[nonpositive_availability] = np.nan

    return stress


def monthly_to_annual_sum(monthly: pd.DataFrame) -> pd.DataFrame:
    """Sum monthly values to annual totals for each HUC8."""
    annual = pd.DataFrame(index=monthly.index)
    for year in YEARS:
        annual[str(year)] = monthly[year_columns(year)].sum(axis=1, min_count=1)
    return annual


def compute_annual_stress(
    monthly_withdrawal: pd.DataFrame,
    monthly_availability: pd.DataFrame,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Calculate annual withdrawal, availability, and annual stress."""
    annual_withdrawal = monthly_to_annual_sum(monthly_withdrawal)
    annual_availability = monthly_to_annual_sum(monthly_availability)
    annual_stress = pd.DataFrame(index=annual_withdrawal.index, columns=annual_withdrawal.columns, dtype=float)

    for year in annual_withdrawal.columns:
        numerator = annual_withdrawal[year].astype(float)
        denominator = annual_availability[year].astype(float)
        values = pd.Series(np.nan, index=annual_withdrawal.index, dtype=float)
        positive = denominator > 0
        values[positive] = numerator[positive] / denominator[positive]

        if TREAT_NONPOSITIVE_AVAILABLE_AS_INF:
            values[(~positive) & (numerator > 0)] = np.inf
            values[(~positive) & (numerator <= 0)] = 0.0

        annual_stress[year] = values

    return annual_withdrawal, annual_availability, annual_stress


def year_columns(year: int) -> List[str]:
    """Return the 12 monthly columns for one modeled year."""
    if year not in YEARS:
        raise ValueError(f"year must be one of {YEARS}; received {year}")
    return [f"{year}_{month:02d}" for month in MONTHS]


def national_average_monthly_stress(
    monthly_withdrawal: pd.DataFrame,
    monthly_availability: pd.DataFrame,
    year: int,
) -> float:
    """
    Calculate the mean of 12 national monthly withdrawal/availability ratios.
    """
    columns = year_columns(year)
    national_withdrawal = monthly_withdrawal[columns].sum(axis=0, min_count=1)
    national_availability = monthly_availability[columns].sum(axis=0, min_count=1)

    monthly_ratios: List[float] = []
    for column in columns:
        withdrawal = national_withdrawal[column]
        availability = national_availability[column]
        if pd.isna(withdrawal) or pd.isna(availability):
            monthly_ratios.append(np.nan)
        elif availability > 0:
            monthly_ratios.append(float(withdrawal / availability))
        elif TREAT_NONPOSITIVE_AVAILABLE_AS_INF and withdrawal > 0:
            monthly_ratios.append(np.inf)
        else:
            monthly_ratios.append(np.nan)

    finite = np.asarray(monthly_ratios, dtype=float)
    finite = finite[np.isfinite(finite)]
    return float(np.mean(finite)) if finite.size else np.nan


def count_highly_stressed_basins(monthly_stress: pd.DataFrame, year: int) -> int:
    """Count HUC8 basins with at least one month at stress >= 0.4."""
    return int((monthly_stress[year_columns(year)] >= HIGH_STRESS_THRESHOLD).any(axis=1).sum())


def count_highly_stressed_basin_months(monthly_stress: pd.DataFrame, year: int) -> int:
    """Count all HUC8-month observations with stress >= 0.4."""
    values = monthly_stress[year_columns(year)] >= HIGH_STRESS_THRESHOLD
    return int(values.to_numpy(dtype=int).sum())


def create_common_other_use(common_2020_baseline: pd.DataFrame) -> pd.DataFrame:
    """Repeat 10% of each 2020 monthly baseline withdrawal in every year."""
    other_use = pd.DataFrame(0.0, index=common_2020_baseline.index, columns=TARGET_COLS)
    for year in YEARS:
        for month in MONTHS:
            source = f"2020_{month:02d}"
            destination = f"{year}_{month:02d}"
            other_use[destination] = (
                common_2020_baseline[source] * OTHER_USE_FRACTION_OF_2020_BASELINE
            )
    return other_use


## 3. Load resources and simulate all cases
The workflow validates the original data path, loads every scenario, harmonizes 2020 runoff, and calculates all 12 scenario–AI combinations.

In [4]:
# =============================================================================
# Source-data loading and scenario simulation
# =============================================================================
def validate_source_paths(base_path: Path) -> None:
    """Fail early with a readable list of missing project resources."""
    missing: List[str] = []

    for scenario_info in SCENARIOS.values():
        runoff_pattern = str(base_path / scenario_info["runoff_folder"] / "*.xlsx")
        if not glob.glob(runoff_pattern):
            missing.append(f"No runoff workbooks matching: {runoff_pattern}")

        for key in ("ir_file", "ps_file", "tp_file"):
            path = base_path / scenario_info[key]
            if not path.exists():
                missing.append(str(path))

        for filename in scenario_info["ai_proj_files"].values():
            path = base_path / filename
            if not path.exists():
                missing.append(str(path))

    if missing:
        formatted = "\n  - ".join(missing)
        raise FileNotFoundError(
            "Some required data resources were not found. Check BASE_PATH and filenames:\n"
            f"  - {formatted}"
        )


def load_raw_data(base_path: Path) -> Tuple[OrderedDict, List[pd.DataFrame]]:
    """Load every SSP-RCP pathway and all four AI cases."""
    raw: OrderedDict = OrderedDict()
    all_runoff_files: List[pd.DataFrame] = []

    for scenario_key, scenario_info in SCENARIOS.items():
        print(f"Loading {scenario_key} ({scenario_info['title']}) ...")

        runoff_paths = sorted(
            Path(path)
            for path in glob.glob(str(base_path / scenario_info["runoff_folder"] / "*.xlsx"))
        )
        runoff_frames = [read_runoff_file(path) for path in runoff_paths]
        all_runoff_files.extend(runoff_frames)
        runoff_average = average_df_list(runoff_frames)

        irrigation = read_csv_monthly(base_path / scenario_info["ir_file"])
        public_supply = read_csv_monthly(base_path / scenario_info["ps_file"])

        thermoelectric: OrderedDict = OrderedDict()
        thermoelectric_path = base_path / scenario_info["tp_file"]
        for case_name, case_info in AI_CASES.items():
            thermoelectric[case_name] = read_excel_monthly(
                thermoelectric_path,
                case_info["tp_sheet"],
                fill_missing_cols=0.0,
                aggregate="sum",
            )

        onsite_ai: OrderedDict = OrderedDict()
        onsite_ai["Baseline"] = pd.DataFrame(
            0.0,
            index=runoff_average.index,
            columns=TARGET_COLS,
        )
        for case_name in ("Low-Case", "Mid-Case", "High-Case"):
            onsite_ai[case_name] = read_excel_monthly(
                base_path / scenario_info["ai_proj_files"][case_name],
                "AI_Onsite_Water_Usage",
                fill_missing_cols=0.0,
                aggregate="sum",
            )

        raw[scenario_key] = {
            "runoff_average": runoff_average,
            "irrigation": irrigation,
            "public_supply": public_supply,
            "thermoelectric": thermoelectric,
            "onsite_ai": onsite_ai,
        }

    return raw, all_runoff_files


def apply_common_2020_availability(
    raw: OrderedDict,
    all_runoff_files: Sequence[pd.DataFrame],
) -> pd.DataFrame:
    """Create and apply the notebook's common 2020 runoff baseline."""
    common_all_year_runoff = average_df_list(all_runoff_files)
    common_2020_availability = common_all_year_runoff[COLS_2020].copy()

    for scenario_data in raw.values():
        runoff = scenario_data["runoff_average"].copy()
        union_index = sorted(set(runoff.index).union(common_2020_availability.index))
        runoff = runoff.reindex(index=union_index, columns=TARGET_COLS)
        common_2020 = common_2020_availability.reindex(index=union_index, columns=COLS_2020)
        for column in COLS_2020:
            runoff[column] = common_2020[column]
        scenario_data["runoff_average"] = runoff

    return common_2020_availability


def build_common_other_use(raw: OrderedDict) -> pd.DataFrame:
    """Create the common 10% Other Use profile from baseline 2020 demand."""
    baseline_2020_frames: List[pd.DataFrame] = []

    for scenario_data in raw.values():
        irrigation = scenario_data["irrigation"]
        public_supply = scenario_data["public_supply"]
        baseline_thermoelectric = scenario_data["thermoelectric"]["Baseline"]

        union_index = sorted(
            set().union(
                irrigation.index,
                public_supply.index,
                baseline_thermoelectric.index,
            )
        )
        baseline_without_other = (
            align_to_index(irrigation, union_index, fill_value=0.0)
            + align_to_index(public_supply, union_index, fill_value=0.0)
            + align_to_index(baseline_thermoelectric, union_index, fill_value=0.0)
        )

        expanded = pd.DataFrame(
            np.nan,
            index=baseline_without_other.index,
            columns=TARGET_COLS,
        )
        expanded[COLS_2020] = baseline_without_other[COLS_2020]
        baseline_2020_frames.append(expanded)

    common_2020_baseline = average_df_list(baseline_2020_frames)[COLS_2020]
    return create_common_other_use(common_2020_baseline)


def simulate_all_cases(
    raw: OrderedDict,
    common_other_use: pd.DataFrame,
) -> Tuple[OrderedDict, OrderedDict]:
    """Calculate component and stress results for all 12 scenario/case pairs."""
    results: OrderedDict = OrderedDict()
    components: OrderedDict = OrderedDict()

    for scenario_key, scenario_data in raw.items():
        print(f"Calculating HUC8 stress for {scenario_key} ...")

        runoff_raw = scenario_data["runoff_average"]
        irrigation_raw = scenario_data["irrigation"]
        public_supply_raw = scenario_data["public_supply"]
        thermoelectric_raw = scenario_data["thermoelectric"]
        onsite_ai_raw = scenario_data["onsite_ai"]

        union_index = sorted(
            set().union(
                runoff_raw.index,
                irrigation_raw.index,
                public_supply_raw.index,
                common_other_use.index,
                *(frame.index for frame in thermoelectric_raw.values()),
                *(frame.index for frame in onsite_ai_raw.values()),
            )
        )

        runoff = align_to_index(runoff_raw, union_index, fill_value=None)
        irrigation = align_to_index(irrigation_raw, union_index, fill_value=0.0)
        public_supply = align_to_index(public_supply_raw, union_index, fill_value=0.0)
        other_use = align_to_index(common_other_use, union_index, fill_value=0.0)
        thermoelectric = OrderedDict(
            (
                case_name,
                align_to_index(frame, union_index, fill_value=0.0),
            )
            for case_name, frame in thermoelectric_raw.items()
        )
        onsite_ai = OrderedDict(
            (
                case_name,
                align_to_index(frame, union_index, fill_value=0.0),
            )
            for case_name, frame in onsite_ai_raw.items()
        )

        components[scenario_key] = {
            "availability": runoff,
            "irrigation": irrigation,
            "public_supply": public_supply,
            "other_use": other_use,
            "thermoelectric": thermoelectric,
            "onsite_ai": onsite_ai,
        }

        results[scenario_key] = OrderedDict()
        for case_name in AI_CASES:
            total_monthly_withdrawal = (
                irrigation
                + public_supply
                + thermoelectric[case_name]
                + onsite_ai[case_name]
                + other_use
            )
            monthly_stress = compute_stress(total_monthly_withdrawal, runoff)
            annual_withdrawal, annual_availability, annual_stress = compute_annual_stress(
                total_monthly_withdrawal,
                runoff,
            )

            results[scenario_key][case_name] = {
                "monthly_withdrawal": total_monthly_withdrawal,
                "monthly_availability": runoff,
                "monthly_stress": monthly_stress,
                "annual_withdrawal": annual_withdrawal,
                "annual_availability": annual_availability,
                "annual_stress": annual_stress,
            }

    return results, components


def run_simulation(base_path: Path) -> Tuple[OrderedDict, OrderedDict]:
    """Complete data-loading and HUC8 simulation workflow."""
    validate_source_paths(base_path)
    raw, all_runoff_files = load_raw_data(base_path)
    apply_common_2020_availability(raw, all_runoff_files)
    common_other_use = build_common_other_use(raw)
    return simulate_all_cases(raw, common_other_use)


## 4. Extract and print demonstration results
The following functions prepare national summaries, ranked HUC8 tables, AI-versus-baseline comparisons, and a monthly basin extract.

In [5]:
# =============================================================================
# Printed result extraction
# =============================================================================
def build_national_summary(results: Mapping, year: int) -> pd.DataFrame:
    """Build a compact summary for all SSP-RCP and AI cases."""
    rows: List[Dict[str, object]] = []

    for scenario_key, scenario_results in results.items():
        for case_name, result in scenario_results.items():
            columns = year_columns(year)
            total_withdrawal_km3 = (
                result["monthly_withdrawal"][columns].sum(axis=1, min_count=1).sum() / 1e9
            )
            rows.append(
                {
                    "Scenario": scenario_key,
                    "SSP-RCP": SCENARIOS[scenario_key]["title"],
                    "AI case": AI_CASES[case_name]["label"],
                    "Year": year,
                    "Withdrawal (km3/yr)": total_withdrawal_km3,
                    "National mean monthly stress": national_average_monthly_stress(
                        result["monthly_withdrawal"],
                        result["monthly_availability"],
                        year,
                    ),
                    "HUC8 basins >=0.4 in any month": count_highly_stressed_basins(
                        result["monthly_stress"], year
                    ),
                    "HUC8 basin-months >=0.4": count_highly_stressed_basin_months(
                        result["monthly_stress"], year
                    ),
                }
            )

    return pd.DataFrame(rows)


def build_huc8_table(result: Mapping[str, pd.DataFrame], year: int) -> pd.DataFrame:
    """Create one annual/monthly diagnostic table for every HUC8."""
    columns = year_columns(year)
    monthly_stress = result["monthly_stress"][columns]

    table = pd.DataFrame(index=monthly_stress.index)
    table.index.name = "HUC8"
    table["Withdrawal_m3_per_year"] = result["annual_withdrawal"][str(year)]
    table["Availability_m3_per_year"] = result["annual_availability"][str(year)]
    table["Annual_stress"] = result["annual_stress"][str(year)]
    table["Maximum_monthly_stress"] = monthly_stress.max(axis=1, skipna=True)
    table["Mean_monthly_stress"] = monthly_stress.mean(axis=1, skipna=True)
    table["High_stress_months"] = (monthly_stress >= HIGH_STRESS_THRESHOLD).sum(axis=1)
    table["Peak_month"] = monthly_stress.idxmax(axis=1, skipna=True).str[-2:]
    return table


def build_ai_change_table(
    results: Mapping,
    scenario: str,
    ai_case: str,
    year: int,
) -> pd.DataFrame:
    """Compare one AI case against the same SSP-RCP no-AI baseline."""
    ai_result = results[scenario][ai_case]
    baseline_result = results[scenario]["Baseline"]

    ai_table = build_huc8_table(ai_result, year)
    baseline_table = build_huc8_table(baseline_result, year)

    comparison = ai_table.add_prefix("AI_").join(baseline_table.add_prefix("Baseline_"), how="outer")
    comparison["Annual_stress_change"] = (
        comparison["AI_Annual_stress"] - comparison["Baseline_Annual_stress"]
    )
    comparison["Withdrawal_change_m3_per_year"] = (
        comparison["AI_Withdrawal_m3_per_year"]
        - comparison["Baseline_Withdrawal_m3_per_year"]
    )
    comparison["Additional_high_stress_months"] = (
        comparison["AI_High_stress_months"]
        - comparison["Baseline_High_stress_months"]
    )
    return comparison


def print_dataframe(frame: pd.DataFrame, *, float_digits: int = 4) -> None:
    """Print a dataframe without permanently modifying pandas display settings."""
    with pd.option_context(
        "display.max_columns",
        None,
        "display.width",
        220,
        "display.float_format",
        lambda value: f"{value:,.{float_digits}f}",
    ):
        print(frame.to_string())


def print_demo_results(
    results: Mapping,
    *,
    scenario: str,
    ai_case: str,
    year: int,
    top_n: int,
    requested_huc8: Optional[str],
) -> None:
    """Print national and HUC8-level examples without writing any files."""
    print("\n" + "=" * 100)
    print(f"NATIONAL SUMMARY FOR {year}")
    print("=" * 100)
    national = build_national_summary(results, year)
    print_dataframe(national.set_index(["Scenario", "AI case"]), float_digits=4)

    selected_result = results[scenario][ai_case]
    huc8_table = build_huc8_table(selected_result, year)

    print("\n" + "=" * 100)
    print(
        f"TOP {top_n} HUC8 BASINS BY ANNUAL STRESS: "
        f"{SCENARIOS[scenario]['title']}, {AI_CASES[ai_case]['label']}, {year}"
    )
    print("=" * 100)
    top_stress = huc8_table.sort_values("Annual_stress", ascending=False).head(top_n)
    print_dataframe(top_stress, float_digits=4)

    if ai_case != "Baseline":
        comparison = build_ai_change_table(results, scenario, ai_case, year)
        columns = [
            "Baseline_Annual_stress",
            "AI_Annual_stress",
            "Annual_stress_change",
            "Withdrawal_change_m3_per_year",
            "Baseline_High_stress_months",
            "AI_High_stress_months",
            "Additional_high_stress_months",
        ]
        print("\n" + "=" * 100)
        print(
            f"TOP {top_n} HUC8 BASINS BY AI-INDUCED ANNUAL STRESS INCREASE: "
            f"{SCENARIOS[scenario]['title']}, {AI_CASES[ai_case]['label']}, {year}"
        )
        print("=" * 100)
        top_changes = comparison.sort_values("Annual_stress_change", ascending=False).head(top_n)
        print_dataframe(top_changes[columns], float_digits=5)

    # Always print one monthly HUC8 extraction. When REQUESTED_HUC8 is None,
    # select the highest-annual-stress HUC8 in the chosen case automatically.
    if requested_huc8 is None:
        finite_rank = huc8_table["Annual_stress"].replace([np.inf, -np.inf], np.nan).dropna()
        huc8 = str(finite_rank.idxmax()) if not finite_rank.empty else str(huc8_table.index[0])
        selection_note = "automatically selected from the highest annual-stress HUC8"
    else:
        huc8 = standardize_huc8(requested_huc8)
        selection_note = "selected in the configuration cell"

    print("\n" + "=" * 100)
    print(f"EXTRACTED MONTHLY RESULT FOR HUC8 {huc8} ({selection_note})")
    print("=" * 100)

    if huc8 not in selected_result["monthly_stress"].index:
        print(f"HUC8 {huc8} was not found in the aligned simulation dataset.")
        return

    columns = year_columns(year)
    extracted = pd.DataFrame(
        {
            "Withdrawal_m3": selected_result["monthly_withdrawal"].loc[huc8, columns],
            "Availability_m3": selected_result["monthly_availability"].loc[huc8, columns],
            "Stress": selected_result["monthly_stress"].loc[huc8, columns],
        }
    )
    extracted.index.name = "Year_Month"
    print_dataframe(extracted, float_digits=5)


## 5. Notebook execution helper

In [6]:
# =============================================================================
# Notebook execution helper
# =============================================================================
def run_notebook_demo() -> Dict[str, object]:
    """Run the complete simulation, print results, and return reusable tables."""
    if SELECTED_SCENARIO not in SCENARIOS:
        raise ValueError(
            f"SELECTED_SCENARIO must be one of {list(SCENARIOS)}; "
            f"received {SELECTED_SCENARIO!r}."
        )
    if SELECTED_AI_CASE not in AI_CASES:
        raise ValueError(
            f"SELECTED_AI_CASE must be one of {list(AI_CASES)}; "
            f"received {SELECTED_AI_CASE!r}."
        )
    if SELECTED_YEAR not in YEARS:
        raise ValueError(
            f"SELECTED_YEAR must be one of {YEARS}; received {SELECTED_YEAR!r}."
        )
    if not isinstance(TOP_N, int) or TOP_N < 1:
        raise ValueError("TOP_N must be a positive integer.")

    print("HUC8 freshwater-stress GitHub notebook demo")
    print(f"Data folder: {BASE_PATH}")
    print("No figures or output files will be created.\n")

    results, components = run_simulation(BASE_PATH)

    # Print the requested summaries immediately.
    print_demo_results(
        results,
        scenario=SELECTED_SCENARIO,
        ai_case=SELECTED_AI_CASE,
        year=SELECTED_YEAR,
        top_n=TOP_N,
        requested_huc8=REQUESTED_HUC8,
    )

    # Also return common tables so later notebook cells can use them directly.
    national_summary = build_national_summary(results, SELECTED_YEAR)
    selected_huc8_results = build_huc8_table(
        results[SELECTED_SCENARIO][SELECTED_AI_CASE],
        SELECTED_YEAR,
    )
    ai_change_results = None
    if SELECTED_AI_CASE != "Baseline":
        ai_change_results = build_ai_change_table(
            results,
            SELECTED_SCENARIO,
            SELECTED_AI_CASE,
            SELECTED_YEAR,
        )

    return {
        "results": results,
        "components": components,
        "national_summary": national_summary,
        "selected_huc8_results": selected_huc8_results,
        "ai_change_results": ai_change_results,
    }


## 6. Run the simulation
Executing this cell loads the project data, calculates every SSP–RCP/AI case, and immediately prints the requested results. No files are created.

In [7]:
# Run all SSP-RCP and AI cases and print demonstration results.
DEMO_OUTPUT = run_notebook_demo()

# Reusable objects retained in memory for later notebook analysis.
RESULTS = DEMO_OUTPUT["results"]
COMPONENTS = DEMO_OUTPUT["components"]
NATIONAL_SUMMARY = DEMO_OUTPUT["national_summary"]
SELECTED_HUC8_RESULTS = DEMO_OUTPUT["selected_huc8_results"]
AI_CHANGE_RESULTS = DEMO_OUTPUT["ai_change_results"]


HUC8 freshwater-stress GitHub notebook demo
Data folder: E:\2026 Spring\Required Data
No figures or output files will be created.

Loading SSP245 (SSP2-RCP4.5) ...
Loading SSP370 (SSP3-RCP7.0) ...
Loading SSP585 (SSP5-RCP8.5) ...
Calculating HUC8 stress for SSP245 ...
Calculating HUC8 stress for SSP370 ...
Calculating HUC8 stress for SSP585 ...

NATIONAL SUMMARY FOR 2035
                           SSP-RCP  Year  Withdrawal (km3/yr)  National mean monthly stress  HUC8 basins >=0.4 in any month  HUC8 basin-months >=0.4
Scenario AI case                                                                                                                                    
SSP245   No AI         SSP2-RCP4.5  2035             291.1248                        0.2076                             812                     4389
         Low-Case AI   SSP2-RCP4.5  2035             293.3345                        0.2091                             818                     4410
         Mid-Case AI   SSP2-RC

C:\Users\xiaot\AppData\Local\Temp\ipykernel_31716\2087515708.py:51: FutureWarning: The behavior of DataFrame.idxmax with all-NA values, or any-NA and skipna=False, is deprecated. In a future version this will raise ValueError
  table["Peak_month"] = monthly_stress.idxmax(axis=1, skipna=True).str[-2:]
C:\Users\xiaot\AppData\Local\Temp\ipykernel_31716\2087515708.py:51: FutureWarning: The behavior of DataFrame.idxmax with all-NA values, or any-NA and skipna=False, is deprecated. In a future version this will raise ValueError
  table["Peak_month"] = monthly_stress.idxmax(axis=1, skipna=True).str[-2:]
C:\Users\xiaot\AppData\Local\Temp\ipykernel_31716\2087515708.py:51: FutureWarning: The behavior of DataFrame.idxmax with all-NA values, or any-NA and skipna=False, is deprecated. In a future version this will raise ValueError
  table["Peak_month"] = monthly_stress.idxmax(axis=1, skipna=True).str[-2:]
C:\Users\xiaot\AppData\Local\Temp\ipykernel_31716\2087515708.py:51: FutureWarning: The behavio

### Main in-memory results

- `RESULTS[scenario][case]["monthly_stress"]`: monthly HUC8 stress.
- `RESULTS[scenario][case]["annual_stress"]`: annual HUC8 stress.
- `COMPONENTS[scenario]`: availability and withdrawal components.
- `NATIONAL_SUMMARY`: scenario-wide summary for `SELECTED_YEAR`.
- `SELECTED_HUC8_RESULTS`: HUC8 diagnostic table for the selected case.
- `AI_CHANGE_RESULTS`: selected AI case compared with the corresponding no-AI baseline.